In [3]:
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_bounds
import numpy as np
import os
import pandas as pd

In [4]:
gdf = gpd.read_file('/home/jan/Documents/EPFL/project_2/data/input_datasets/Africa_Masolele/Africa_landuse_following_deforestation_training_Polygon.shp')


In [6]:
gdf.loc[gdf.driver.isin(['pasture', 'mining', 'small scale cropland', 'tree crops',
       'other tree cover', 'palm oil', 'rubber', 'large scale cropland',
       'urban', 'water', 'cashew', 'roads'])]

array(['pasture', 'mining', 'small scale cropland', 'tree crops',
       'other tree cover', 'palm oil', 'rubber', 'large scale cropland',
       'urban', 'water', 'cashew', 'roads'], dtype=object)

In [8]:
gdf.LU.unique()

array([ 2.,  3.,  4.,  5.,  6., 12., 13.,  1.,  9., 11., 14., 16.])

In [17]:
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_bounds
import numpy as np
import re
import os

def split_polygons_by_latlon(gdf):
    # Ensure CRS is in degrees (lat/lon)
    gdf = gdf.to_crs(epsg=4326)
    
    # Create a dictionary to hold subsets
    latlon_clusters = {}

    for _, polygon in gdf.iterrows():
        minx, miny, maxx, maxy = polygon.geometry.bounds
        lon_deg = round(minx)
        lat_deg = round(miny)
        key = (lat_deg, lon_deg)
        
        if key not in latlon_clusters:
            latlon_clusters[key] = []
        latlon_clusters[key].append(polygon)

    # Convert lists to GeoDataFrames
    for key in latlon_clusters:
        latlon_clusters[key] = gpd.GeoDataFrame(latlon_clusters[key], columns=gdf.columns, crs=gdf.crs)

    return latlon_clusters


def rasterize_polygons(gdf, out_path, resolution=10, max_size=10000):
    # Check if there are any geometries to rasterize
    if gdf.empty or gdf.geometry.is_empty.all():
        print(f"Skipping {out_path} as it is empty.")
        return
    
    # Ensure CRS is in meters
    gdf = gdf.to_crs(gdf.estimate_utm_crs())
    
    # Get the bounds of the GeoDataFrame
    minx, miny, maxx, maxy = gdf.total_bounds

    # Calculate the dimensions of the raster based on the resolution
    width = int((maxx - minx) / resolution)
    height = int((maxy - miny) / resolution)

    # Skip rasterization if width or height is zero
    if width <= 0 or height <= 0:
        print(f"Skipping {out_path} due to invalid dimensions (width: {width}, height: {height}).")
        return

    # Create a transform for the output raster
    transform = from_bounds(minx, miny, maxx, maxy, width, height)

    # Rasterize the GeoDataFrame
    raster = rasterize(
        [(geom, int(value)) for geom, value in zip(gdf.geometry, gdf['LU'])],
        out_shape=(height, width),
        transform=transform,
        fill=0,
        dtype='uint8'
    )

    # Split large rasters if necessary
    if width > max_size or height > max_size:
        x_splits = max(1, width // max_size + (1 if width % max_size != 0 else 0))
        y_splits = max(1, height // max_size + (1 if height % max_size != 0 else 0))
        for i in range(x_splits):
            for j in range(y_splits):
                x_min = i * max_size
                x_max = min((i + 1) * max_size, width)
                y_min = j * max_size
                y_max = min((j + 1) * max_size, height)
                
                sub_raster = raster[y_min:y_max, x_min:x_max]
                sub_transform = transform * transform.translation(x_min, y_min)
                
                sub_out_path = f"{os.path.splitext(out_path)[0]}_{i}_{j}.tif"
                if np.any(sub_raster):
                    with rasterio.open(
                        sub_out_path,
                        'w',
                        driver='GTiff',
                        height=sub_raster.shape[0],
                        width=sub_raster.shape[1],
                        count=1,
                        dtype=sub_raster.dtype,
                        crs=gdf.crs,
                        transform=sub_transform,
                    ) as dst:
                        dst.write(sub_raster, 1)
                    print(f"Written {sub_out_path}")
    else:
        if np.any(raster):
            with rasterio.open(
                out_path,
                'w',
                driver='GTiff',
                height=raster.shape[0],
                width=raster.shape[1],
                count=1,
                dtype=raster.dtype,
                crs=gdf.crs,
                transform=transform,
            ) as dst:
                dst.write(raster, 1)
            print(f"Written {out_path}")

            
            
def sanitize_filename(name):
    # Replace or remove any problematic characters for filenames
    name = re.sub(r'[^\w\s-]', '', name)
    name = re.sub(r'[-\s]+', '_', name)
    return name

In [18]:


# Example usage
# polygons_gdf: GeoDataFrame of polygons to split

latlon_clusters = split_polygons_by_latlon(gdf)


In [20]:
latlon_clusters

{(0,
  30):      geometry_t Sour_90_00 LU_1990 LU_2000   LU                driver  \
 0       Polygon      GE_01      11      11  2.0               pasture   
 1       Polygon      GE_01      11      11  2.0               pasture   
 2       Polygon      GE_01      11      11  2.0               pasture   
 3       Polygon      GE_01      11      11  2.0               pasture   
 100     Polygon      GE_01      11      11  4.0  small scale cropland   
 644     Polygon      GE_03      11      30  4.0  small scale cropland   
 645     Polygon      GE_03      11      30  4.0  small scale cropland   
 5044    Polygon    GLSE_00      11      11  2.0               pasture   
 5045    Polygon    GLSE_00      11      11  2.0               pasture   
 5567    Polygon    GLSE_00      11      11  4.0  small scale cropland   
 
           Tile CHLU_00_05 Source_00_ LU_2005  ... Source_90_ D_90_00  \
 0     N00_E030       1112       None      12  ...       None     100   
 1     N00_E030       1112 

In [19]:
len(latlon_clusters.keys())

522

In [24]:

# Rasterize each subset
for (lat, lon), subset_gdf in latlon_clusters.items():
    cleaned_filename = sanitize_filename(f"africa_{lat}_{lon}_raster")
    print(f"Processing cluster at lat {lat}, lon {lon}")
    rasterize_polygons(subset_gdf, (f'../data/tmp/{cleaned_filename}.tif'))

Processing cluster at lat 0, lon 30
Written ../data/tmp/africa_0_30_raster.tif
Processing cluster at lat 11, lon 12
Written ../data/tmp/africa_11_12_raster.tif
Processing cluster at lat 11, lon 20
Written ../data/tmp/africa_11_20_raster.tif
Processing cluster at lat -24, lon 33
Written ../data/tmp/africa__24_33_raster.tif
Processing cluster at lat 13, lon -15
Written ../data/tmp/africa_13__15_raster.tif
Processing cluster at lat 11, lon -9
Written ../data/tmp/africa_11__9_raster.tif
Processing cluster at lat 10, lon 2
Written ../data/tmp/africa_10_2_raster.tif
Processing cluster at lat 12, lon 3
Written ../data/tmp/africa_12_3_raster.tif
Processing cluster at lat -25, lon 46
Written ../data/tmp/africa__25_46_raster.tif
Processing cluster at lat 12, lon -10
Written ../data/tmp/africa_12__10_raster.tif
Processing cluster at lat 12, lon -9
Written ../data/tmp/africa_12__9_raster.tif
Processing cluster at lat 8, lon 17
Written ../data/tmp/africa_8_17_raster.tif
Processing cluster at lat 12

KeyboardInterrupt: 

In [24]:
country_subsets_f = {k:v for k,v in country_subsets.items() if not v.empty}

In [52]:
# Rasterize each subset
for country, subset_gdf in country_subsets_f.items():
    print(country)
    sanitized_country_name = sanitize_filename(country)
    rasterize_polygons(subset_gdf, f'../data/tmp/manual_{sanitized_country_name}_v3.tif')

Colombia
Written ../data/tmp/manual_Colombia_v3_0_6.tif
Written ../data/tmp/manual_Colombia_v3_1_5.tif
Written ../data/tmp/manual_Colombia_v3_4_0.tif
Written ../data/tmp/manual_Colombia_v3_6_0.tif
Written ../data/tmp/manual_Colombia_v3_7_0.tif
Costa Rica
Written ../data/tmp/manual_Costa_Rica_v3.tif
Honduras
Written ../data/tmp/manual_Honduras_v3.tif
Venezuela
Written ../data/tmp/manual_Venezuela_v3.tif
Puerto Rico
Written ../data/tmp/manual_Puerto_Rico_v3.tif
Côte d'Ivoire
Written ../data/tmp/manual_Côte_dIvoire_v3_0_0.tif
Written ../data/tmp/manual_Côte_dIvoire_v3_3_0.tif
Written ../data/tmp/manual_Côte_dIvoire_v3_4_1.tif
Burundi
Written ../data/tmp/manual_Burundi_v3.tif
Ethiopia
Written ../data/tmp/manual_Ethiopia_v3.tif
Uganda
Written ../data/tmp/manual_Uganda_v3_0_2.tif
Written ../data/tmp/manual_Uganda_v3_4_0.tif
Written ../data/tmp/manual_Uganda_v3_5_0.tif


In [30]:
import re